In [7]:
import os, time, numpy as np, gymnasium as gym, torch
from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.utils import get_linear_fn

os.makedirs("logs_dqn", exist_ok=True)
os.makedirs("logs_ppo", exist_ok=True)
os.makedirs("logs_a2c", exist_ok=True)

print("CUDA:", torch.cuda.is_available())

CUDA: False


Para cada una de las opciones vamos a probar con unas mini variaciones (aunque se tengan los hiperparametros tuneadis)

In [8]:
import statistics

def mini_crosval(algo_name, steps, log_dir, LRs, algo_class, hiperparametros, lr_schedule="constant"):
    LRs = LRs
    SEEDS = [0, 42]
    cv_steps = steps
    cv_dir = f"logs_{algo_name}{log_dir}"

    os.makedirs(cv_dir, exist_ok=True)
    sweep_results = {}

    for lr in LRs:
        for seed in SEEDS:
            run_name = f"lr{lr:.0e}_s{seed}"
            run_dir  = f"{cv_dir}/{run_name}"
            os.makedirs(run_dir, exist_ok=True)

            env = Monitor(gym.make("LunarLander-v3"), filename=f"{run_dir}/train")
            eval_env = Monitor(gym.make("LunarLander-v3"), filename=f"{run_dir}/eval")
            
            # MISMA semilla eval todas las runs
            eval_env.reset(seed=1234)

            # constante o lineal (decay a 0))
            if lr_schedule == "constant":
                lr_value = lr
            elif lr_schedule == "linear":
                lr_value = get_linear_fn(lr, 0.0, 1.0)

            model = algo_class(
                "MlpPolicy", env, verbose=0, seed=seed, device="cpu",
                learning_rate=lr_value,
                **hiperparametros
            )

            t0 = time.time()
            model.learn(total_timesteps=cv_steps)
            dt = (time.time() - t0) / 60

            mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
            sweep_results[(lr, seed)] = {"mean": mean_r, "std": std_r, "time_min": dt}
            print(f"{run_name:18s} reward = {mean_r:7.1f} +- {std_r:5.1f}   ({dt:4.1f} min)")
            env.close(); eval_env.close()
    
    print(f"\nResumen {algo_name} ({lr_schedule}) por LR (media semillas)")
    agg = {}
    for lr in LRs:
        means = [sweep_results[(lr, s)]["mean"] for s in SEEDS]
        m, s = statistics.mean(means), (statistics.stdev(means) if len(means) > 1 else 0.0)
        agg[lr] = (m, s)
        print(f"lr={lr:.0e} reward = {m:7.1f} +- {s:5.1f}")

    # Gana mejor media pero penaliza mucha varianza
    best_lr = max(agg, key=lambda l: agg[l][0] - agg[l][1])
    print(f"\nGanador: lr = {best_lr:.0e} (mean-std = {agg[best_lr][0]-agg[best_lr][1]:.1f})")

    return best_lr

In [9]:
best_lr_dqn = mini_crosval(
    algo_name="dqn",
    steps=50_000,
    log_dir="_sweep",
    LRs=[3e-4, 6.3e-4, 1e-3],
    algo_class=DQN,
    hiperparametros=dict(
        batch_size=128,
        buffer_size=50_000,
        learning_starts=0,
        gamma=0.99,
        target_update_interval=250,
        train_freq=4,
        gradient_steps=-1,
        exploration_fraction=0.12,
        exploration_final_eps=0.1,
        policy_kwargs=dict(net_arch=[256, 256]),
    ),
)

# PPO Zoo no especifica LR
best_lr_ppo = mini_crosval(
    algo_name="ppo",
    steps=100_000,
    log_dir="_sweep",
    LRs=[1e-4, 3e-4, 6e-4],
    algo_class=PPO,
    hiperparametros=dict(
        n_steps=1024,
        batch_size=64,
        n_epochs=4,
        gamma=0.999,
        gae_lambda=0.98,
        ent_coef=0.01,
    ),
)

# A2C constante (sin schedule)
best_lr_a2c = mini_crosval(
    algo_name="a2c",
    steps=100_000,
    log_dir="_sweep",
    LRs=[3e-4, 5e-4, 8e-4],
    algo_class=A2C,
    hiperparametros=dict(
        n_steps=5,
        gamma=0.995,
        ent_coef=1e-5,
        use_rms_prop=True,
    ),
)

# A2C con LR lineal (como el Zoo lin_0.00083)
best_lr_a2c_lin = mini_crosval(
    algo_name="a2c",
    steps=100_000,
    log_dir="_sweep_lin",
    LRs=[5e-4, 8e-4, 1e-3],
    algo_class=A2C,
    hiperparametros=dict(
        n_steps=5,
        gamma=0.995,
        ent_coef=1e-5,
        use_rms_prop=True,
    ),
    lr_schedule="linear",
)

lr3e-04_s0         reward =   -48.9 +-  23.4   ( 3.5 min)
lr3e-04_s42        reward =    -7.3 +-  30.5   ( 3.5 min)
lr6e-04_s0         reward =   256.6 +-  26.6   ( 3.5 min)
lr6e-04_s42        reward =   131.6 +-  43.2   ( 3.6 min)
lr1e-03_s0         reward =   -90.8 +- 101.0   ( 3.6 min)
lr1e-03_s42        reward =   128.4 +- 109.8   ( 3.5 min)

Resumen dqn (constant) por LR (media semillas)
lr=3e-04 reward =   -28.1 +-  29.4
lr=6e-04 reward =   194.1 +-  88.4
lr=1e-03 reward =    18.8 +- 155.0

Ganador: lr = 6e-04 (mean-std = 105.7)
lr1e-04_s0         reward =  -527.0 +- 191.1   ( 1.2 min)
lr1e-04_s42        reward = -3800.8 +- 1546.0   ( 1.2 min)
lr3e-04_s0         reward =  -294.7 +-  89.0   ( 1.2 min)
lr3e-04_s42        reward =  -710.0 +- 123.9   ( 1.3 min)
lr6e-04_s0         reward =    55.4 +- 115.4   ( 1.3 min)
lr6e-04_s42        reward =   -45.8 +-  87.5   ( 1.2 min)

Resumen ppo (constant) por LR (media semillas)
lr=1e-04 reward = -2163.9 +- 2314.9
lr=3e-04 reward =  -502.4 

/home/jorge/Escritorio/MASTER/AprendizajeRefuerzo/.venv/lib/python3.11/site-packages/stable_baselines3/common/utils.py:193: UserWarning: get_linear_fn() is deprecated, please use LinearSchedule() instead
  warnings.warn("get_linear_fn() is deprecated, please use LinearSchedule() instead")


lr5e-04_s0         reward =    80.7 +- 121.8   ( 2.6 min)
lr5e-04_s42        reward =    61.8 +- 148.7   ( 2.4 min)
lr8e-04_s0         reward =  -155.2 +-  33.3   ( 2.3 min)
lr8e-04_s42        reward =     6.4 +- 177.7   ( 2.4 min)
lr1e-03_s0         reward =   -80.6 +-  98.8   ( 2.5 min)
lr1e-03_s42        reward =  -120.9 +-  50.2   ( 2.6 min)

Resumen a2c (linear) por LR (media semillas)
lr=5e-04 reward =    71.3 +-  13.3
lr=8e-04 reward =   -74.4 +- 114.3
lr=1e-03 reward =  -100.8 +-  28.5

Ganador: lr = 5e-04 (mean-std = 57.9)


In [10]:
env = Monitor(gym.make("LunarLander-v3"), filename="logs_dqn/train")
eval_env = Monitor(gym.make("LunarLander-v3"), filename="logs_dqn/eval")

model = DQN(
    "MlpPolicy", env, verbose=0, seed=42, device="cpu",
    learning_rate=best_lr_dqn,
    batch_size=128,
    buffer_size=50_000,
    learning_starts=0,
    gamma=0.99,
    target_update_interval=250,
    train_freq=4,
    gradient_steps=-1,
    exploration_fraction=0.12,
    exploration_final_eps=0.1,
    policy_kwargs=dict(net_arch=[256, 256]),
)

eval_cb = EvalCallback(eval_env, best_model_save_path="logs_dqn/best",
                       log_path="logs_dqn/eval", eval_freq=5000,
                       n_eval_episodes=5, deterministic=True, verbose=0)

t0 = time.time()
model.learn(total_timesteps=100_000, callback=eval_cb)
print(f"DQN entrenamiento: {(time.time()-t0)/60:.1f} min")

mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
print(f"DQN reward 20 episodios: {mean_r:.1f} +- {std_r:.1f}")
model.save("logs_dqn/dqn_lunarlander")

env.close()
eval_env.close()

DQN entrenamiento: 8.0 min
DQN reward 20 episodios: 211.6 +- 67.9


In [11]:
# PPO Zoo n_envs=16 n_timesteps=1e6
train_env = make_vec_env("LunarLander-v3", n_envs=16, monitor_dir="logs_ppo")
eval_env = Monitor(gym.make("LunarLander-v3"), filename="logs_ppo/eval")

model = PPO(
    "MlpPolicy", train_env, verbose=0, seed=42, device="cpu",
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    learning_rate=best_lr_ppo,
)

eval_cb = EvalCallback(eval_env, best_model_save_path="logs_ppo/best",
                       log_path="logs_ppo/eval", eval_freq=5000,
                       n_eval_episodes=5, deterministic=True, verbose=0)

t0 = time.time()
model.learn(total_timesteps=1_000_000, callback=eval_cb)
print(f"PPO entrenamiento: {(time.time()-t0)/60:.1f} min")

mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
print(f"PPO reward 20 episodios: {mean_r:.1f} +- {std_r:.1f}")
model.save("logs_ppo/ppo_lunarlander")

train_env.close()
eval_env.close()

PPO entrenamiento: 16.6 min
PPO reward 20 episodios: 273.0 +- 22.8


In [12]:
# A2C Zoo: n_envs=8, n_timesteps=2e5, ent_coef=1e-5, learning_rate=lin_<best>
train_env = make_vec_env("LunarLander-v3", n_envs=8, monitor_dir="logs_a2c")
eval_env = Monitor(gym.make("LunarLander-v3"), filename="logs_a2c/eval")

model = A2C(
    "MlpPolicy", train_env, verbose=0, seed=42, device="cpu",
    n_steps=5,
    gamma=0.995,
    ent_coef=1e-5,
    learning_rate=get_linear_fn(best_lr_a2c_lin, 0.0, 1.0),
    use_rms_prop=True,
)

eval_cb = EvalCallback(eval_env, best_model_save_path="logs_a2c/best",
                       log_path="logs_a2c/eval", eval_freq=5000,
                       n_eval_episodes=5, deterministic=True, verbose=0)

t0 = time.time()
model.learn(total_timesteps=200_000, callback=eval_cb)
print(f"A2C entrenamiento: {(time.time()-t0)/60:.1f} min")

mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
print(f"A2C reward 20 episodios: {mean_r:.1f} +- {std_r:.1f}")
model.save("logs_a2c/a2c_lunarlander")

train_env.close()
eval_env.close()

A2C entrenamiento: 2.9 min
A2C reward 20 episodios: -91.8 +- 173.8
